In [185]:
import os
import sys
import time

# 1. 현재 노트북 파일이 있는 위치를 작업 디렉토리로 설정
current_dir = os.getcwd()
print(f"현재 작업 디렉토리: {current_dir}")

# 2. 만약 경로가 다르면 talkback_lib.py가 있는 폴더를 sys.path에 추가
if current_dir not in sys.path:
    sys.path.append(current_dir)

# 3. 임포트 테스트
try:
    from talkback_lib import A11yAdbClient
    print("✅ talkback_lib 로드 성공!")
except ImportError as e:
    print(f"❌ 여전히 찾을 수 없습니다. 에러 메시지: {e}")


client = A11yAdbClient()

# 1. 히스토리 초기화
print("히스토리를 초기화합니다.")
client.reset_focus_history() 
time.sleep(0.5)

현재 작업 디렉토리: d:\Python test\talkback-a11y-helper
✅ talkback_lib 로드 성공!
히스토리를 초기화합니다.
[default] Focus history has been explicitly reset.


In [183]:
client.move_focus_smart()

'failed'

In [153]:
client.reset_focus_history() 

[default] Focus history has been explicitly reset.


In [ ]:
import subprocess
import re
import os

def save_clean_log(filename="result_log.txt"):
    print("⏳ 1단계: 스마트폰 내부에서 원본 로그 생성 중...")
    # 윈도우가 간섭할 수 없도록 안드로이드 내부에 직접 파일 생성
    subprocess.run(["adb", "shell", "logcat -d -s A11Y_HELPER > /sdcard/temp_log.txt"])
    
    print("⏳ 2단계: PC로 원본 파일 가져오기...")
    subprocess.run(["adb", "pull", "/sdcard/temp_log.txt", "temp_log.txt"], capture_output=True)
    subprocess.run(["adb", "shell", "rm", "/sdcard/temp_log.txt"]) # 기기 용량 확보를 위해 삭제
    
    print("⏳ 3단계: 특수문자 제거 및 윈도우 호환 포맷으로 변환 중...")
    # 파일을 '바이트(바이너리)' 모드로 읽어 윈도우의 개입을 100% 차단
    with open("temp_log.txt", "rb") as f:
        raw_bytes = f.read()
        
    # 완벽한 UTF-8로 해석
    raw_text = raw_bytes.decode("utf-8", errors="ignore")
    
    # 지저분한 터미널 색상 코드(ANSI) 싹 지우기
    clean_text = re.sub(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])', '', raw_text)
    
    # 메모장이 한자인 줄 착각하지 못하게 BOM(utf-8-sig) 도장을 찍어서 최종 저장
    with open(filename, "w", encoding="utf-8-sig") as f:
        f.write(clean_text)
        
    # PC에 남은 임시 파일 청소
    if os.path.exists("temp_log.txt"):
        os.remove("temp_log.txt")
        
    print(f"✅ 추출 완료! 이제 폴더의 '{filename}'을 더블클릭해서 열어보세요.")

# 함수 실행
save_clean_log("result_log.txt")

In [152]:
import subprocess
import re

def save_clean_log(filename="result_log.txt", clear_first=False):
    if clear_first:
        print("⏳ 로그 버퍼 초기화 중...")
        subprocess.run(["adb", "logcat", "-c"], check=False)

    print("⏳ A11Y_HELPER 로그 추출 중...")
    result = subprocess.run(
        ["adb", "logcat", "-d", "-s", "A11Y_HELPER"],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="ignore",
        check=False
    )

    raw_text = result.stdout

    print("⏳ 특수문자 제거 및 윈도우 호환 포맷으로 변환 중...")
    clean_text = re.sub(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])', '', raw_text)

    with open(filename, "w", encoding="utf-8-sig") as f:
        f.write(clean_text)

    print(f"✅ 추출 완료! '{filename}' 파일을 확인하세요.")

# 사용 예시
# 1) 새 로그만 보고 싶으면 먼저 logcat 비우고 재현 후 아래 실행
save_clean_log("result_log.txt")

⏳ A11Y_HELPER 로그 추출 중...
⏳ 특수문자 제거 및 윈도우 호환 포맷으로 변환 중...
✅ 추출 완료! 'result_log.txt' 파일을 확인하세요.


In [184]:
client._focus_first_node()

TypeError: A11yAdbClient._focus_first_node() missing 2 required positional arguments: 'dev' and 'nodes'